# Principal Component Analysis

In [ ]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Libraries imported successfully.")

In [ ]:
from pathlib import Path

base_path = Path("../data/processed")

customer_file = base_path / "customer_level_data.csv"
customer_level_data = pd.read_csv(customer_file)

print("Customer-level dataset loaded successfully.")
print("Dataset shape:", customer_level_data.shape)

print("\nAvailable columns:")
for column in customer_level_data.columns:
    print(column)

In [ ]:
# Display the first five customer records
display(customer_level_data.head())

In [ ]:
# ============================================================
# 8. INSPECT ALL CUSTOMER-LEVEL VARIABLES
# ============================================================

print("Dataset shape:", customer_level_data.shape)
print("\nAll available columns:\n")

for i, column in enumerate(customer_level_data.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# ============================================================
# 9. SELECT VARIABLES FOR PCA
# ============================================================

# Select one representative version of each customer behaviour.
#
# Log-transformed versions are used for highly skewed variables
# where these transformations were already created during the
# earlier exploratory analysis.
#
# CustomerID and purchase dates are excluded because they are
# identifiers/time stamps rather than behavioural measurements.
#
# We also avoid including both the original and log-transformed
# version of the same variable because that would duplicate
# information in the PCA.

pca_features = [
    'Log_Frequency',
    'Log_Monetary',
    'Log_AverageOrderValue',
    'Log_AverageBasketQuantity',
    'Log_AverageUniqueProducts',
    'Recency',
    'CustomerTenure',
    'Log_ProductDiversity',
    'Log_ReturnTransactions',
    'Log_ReturnRate'
]

X_pca = customer_level_data[pca_features].copy()

print("PCA dataset shape:", X_pca.shape)

print("\nSelected PCA variables:")
for i, feature in enumerate(pca_features, start=1):
    print(f"{i}. {feature}")

In [ ]:
# ============================================================
# 10. CHECK PCA DATA QUALITY
# ============================================================

print("Missing values:")
print(X_pca.isnull().sum())

print("\nInfinite values:")
print(
    pd.Series(
        np.isinf(X_pca).sum(),
        index=X_pca.columns
    )
)

print("\nDataset shape:")
print(X_pca.shape)

In [ ]:
# ============================================================
# 11. CORRELATION ANALYSIS BEFORE PCA
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Pearson correlation is appropriate here because PCA
# examines linear relationships between variables.

correlation_matrix = X_pca.corr(method='pearson')

print("Correlation Matrix:")
display(correlation_matrix.round(2))


# Create a correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))

im = ax.imshow(
    correlation_matrix,
    cmap='coolwarm',
    vmin=-1,
    vmax=1
)

# Use shorter labels to make the graph easier to read.
short_labels = [
    'Frequency',
    'Monetary',
    'AOV',
    'Basket Quantity',
    'Basket Variety',
    'Recency',
    'Tenure',
    'Product Diversity',
    'Return Transactions',
    'Return Rate'
]

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))

ax.set_xticklabels(
    short_labels,
    rotation=45,
    ha='right'
)

ax.set_yticklabels(short_labels)

# Show the correlation coefficient inside each cell.
for i in range(len(short_labels)):
    for j in range(len(short_labels)):
        ax.text(
            j, i,
            f'{correlation_matrix.iloc[i, j]:.2f}',
            ha='center',
            va='center',
            fontsize=8
        )

plt.colorbar(
    im,
    ax=ax,
    label='Pearson Correlation'
)

plt.title(
    'Correlation Between Selected Customer Behaviour Variables'
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 12. IDENTIFY THE STRONGEST CORRELATIONS
# ============================================================

# Keep only the upper half of the correlation matrix
# to avoid listing each variable pair twice.

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    'Variable 1',
    'Variable 2',
    'Correlation'
]

# Sort by absolute correlation strength.
correlation_pairs['Absolute Correlation'] = (
    correlation_pairs['Correlation'].abs()
)

correlation_pairs = correlation_pairs.sort_values(
    'Absolute Correlation',
    ascending=False
)

print("Top 10 strongest correlation pairs:")

display(
    correlation_pairs.head(10).round(3)
)

In [ ]:
# ============================================================
# 13. STANDARDIZE VARIABLES BEFORE PCA
# ============================================================

from sklearn.preprocessing import StandardScaler

# PCA is sensitive to variable scale.
# Standardization gives each selected variable a mean of 0
# and a standard deviation of approximately 1, preventing
# variables with larger numerical scales from dominating PCA.

scaler_pca = StandardScaler()

X_pca_scaled = scaler_pca.fit_transform(X_pca)

# Convert back to a DataFrame so that we can easily
# inspect the standardized values.
X_pca_scaled_df = pd.DataFrame(
    X_pca_scaled,
    columns=pca_features
)

print("Standardized dataset shape:", X_pca_scaled_df.shape)

print("\nMean after standardization:")
display(X_pca_scaled_df.mean().round(3))

print("\nStandard deviation after standardization:")
display(X_pca_scaled_df.std().round(3))

In [ ]:
# ============================================================
# 14. RUN PRINCIPAL COMPONENT ANALYSIS
# ============================================================

from sklearn.decomposition import PCA

# Keep all 10 components initially.
# We first examine how much variance each component explains
# before deciding whether dimensionality reduction is useful.

pca = PCA()

X_pca_components = pca.fit_transform(X_pca_scaled)

print("PCA completed successfully.")
print("Number of original variables:", X_pca.shape[1])
print("Number of principal components:", pca.n_components_)

In [ ]:
# ============================================================
# 15. CALCULATE EXPLAINED VARIANCE
# ============================================================

# Percentage of total variance explained by each component
explained_variance = pca.explained_variance_ratio_

# Cumulative percentage explained as components are added
cumulative_variance = np.cumsum(explained_variance)

pca_variance_table = pd.DataFrame({
    'Principal Component': [
        f'PC{i+1}'
        for i in range(len(explained_variance))
    ],

    'Explained Variance (%)':
        explained_variance * 100,

    'Cumulative Variance (%)':
        cumulative_variance * 100
})

display(pca_variance_table.round(2))

In [ ]:
# ============================================================
# 16. COMPONENTS REQUIRED FOR 80% AND 90% VARIANCE
# ============================================================

components_80 = np.argmax(
    cumulative_variance >= 0.80
) + 1

components_90 = np.argmax(
    cumulative_variance >= 0.90
) + 1

print(
    "Components needed for at least 80% variance:",
    components_80
)

print(
    "Components needed for at least 90% variance:",
    components_90
)

In [ ]:
# ============================================================
# 17. CALCULATE PCA COMPONENT LOADINGS
# ============================================================

# PCA loadings show how strongly each original customer
# variable contributes to each principal component.
#
# Positive and negative signs show the direction of the
# relationship, while larger absolute values indicate
# stronger contributions.

loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)

print("PCA Component Loadings:")
display(loadings.round(3))

In [ ]:
# ============================================================
# 17. CREATE PC1 vs PC2 CUSTOMER SCATTER PLOT
# ============================================================

# Put the PCA scores into a DataFrame
pca_scores = pd.DataFrame(
    X_pca_components,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)

# Add CustomerID only for reference.
# CustomerID was NOT used to calculate the PCA.
pca_scores['CustomerID'] = customer_level_data['CustomerID'].values

print("PCA score dataset shape:", pca_scores.shape)

display(pca_scores.head())

In [ ]:
# ============================================================
# 18. PLOT CUSTOMERS IN PCA SPACE
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

plt.scatter(
    pca_scores['PC1'],
    pca_scores['PC2'],
    alpha=0.35,
    s=18
)

plt.axhline(
    0,
    linewidth=0.8,
    linestyle='--'
)

plt.axvline(
    0,
    linewidth=0.8,
    linestyle='--'
)

plt.xlabel(
    f'PC1 ({explained_variance[0] * 100:.2f}% variance)'
)

plt.ylabel(
    f'PC2 ({explained_variance[1] * 100:.2f}% variance)'
)

plt.title(
    'Customers in Principal Component Space',
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 19. PCA EXPLAINED VARIANCE GRAPH
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

components = np.arange(1, len(explained_variance) + 1)

fig, ax1 = plt.subplots(figsize=(9, 6))

# ------------------------------------------------------------
# Bars = variance explained by each individual component
# ------------------------------------------------------------
ax1.bar(
    components,
    explained_variance * 100,
    alpha=0.7
)

ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Individual Explained Variance (%)')

ax1.set_xticks(components)
ax1.set_xticklabels(
    [f'PC{i}' for i in components]
)

# ------------------------------------------------------------
# Second axis = cumulative explained variance
# ------------------------------------------------------------
ax2 = ax1.twinx()

ax2.plot(
    components,
    cumulative_variance * 100,
    marker='o',
    linewidth=2
)

ax2.set_ylabel('Cumulative Explained Variance (%)')
ax2.set_ylim(0, 105)

# 80% reference line
ax2.axhline(
    y=80,
    linestyle='--',
    linewidth=1.5,
    label='80% threshold'
)

# 90% reference line
ax2.axhline(
    y=90,
    linestyle=':',
    linewidth=1.5,
    label='90% threshold'
)

# Highlight the important cumulative values
ax2.annotate(
    '4 PCs = 85.54%',
    xy=(4, cumulative_variance[3] * 100),
    xytext=(4.4, 72),
    arrowprops=dict(arrowstyle='->')
)

ax2.annotate(
    '5 PCs = 92.22%',
    xy=(5, cumulative_variance[4] * 100),
    xytext=(5.5, 82),
    arrowprops=dict(arrowstyle='->')
)

ax2.legend(loc='lower right')

plt.title(
    'PCA Explained Variance',
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PCA LOADINGS FOR PC1-PC4
# ============================================================

loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)

loadings_4 = loadings[['PC1', 'PC2', 'PC3', 'PC4']]

display(loadings_4.round(3))

In [ ]:
# ============================================================
# 20. CREATE CUSTOMER VALUE SEGMENTS FOR PCA VISUALISATION
# ============================================================

# Create four customer-value groups using historical Monetary value.
# Quartiles are used so that customers are divided into four
# approximately equal-sized groups.
#
# IMPORTANT:
# These are descriptive historical customer-value segments.
# They are NOT predictions of future Customer Lifetime Value (CLV).

customer_level_data['CustomerValueSegment'] = pd.qcut(
    customer_level_data['Monetary'],
    q=4,
    labels=[
        'Low',
        'Medium',
        'High',
        'Very High'
    ]
)

print("Number of customers in each value segment:")
print(
    customer_level_data['CustomerValueSegment']
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 21. ADD CUSTOMER VALUE SEGMENTS TO PCA SCORES
# ============================================================

pca_scores['CustomerValueSegment'] = (
    customer_level_data['CustomerValueSegment'].values
)

display(pca_scores.head())

In [ ]:
# ============================================================
# 22. PCA SCATTER PLOT BY CUSTOMER VALUE SEGMENT
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 7))

# Colours progress from lower to higher historical customer value
segment_colours = {
    'Low': '#440154',
    'Medium': '#31688e',
    'High': '#35b779',
    'Very High': '#fde725'
}

for segment in ['Low', 'Medium', 'High', 'Very High']:

    subset = pca_scores[
        pca_scores['CustomerValueSegment'] == segment
    ]

    plt.scatter(
        subset['PC1'],
        subset['PC2'],
        label=segment,
        color=segment_colours[segment],
        alpha=0.50,
        s=20
    )

# Reference lines
plt.axhline(
    y=0,
    color='grey',
    linestyle='--',
    linewidth=0.8
)

plt.axvline(
    x=0,
    color='grey',
    linestyle='--',
    linewidth=0.8
)

# Use the actual explained variance from our PCA
plt.xlabel(
    'PC1 (47.11% variance)',
    fontsize=11
)

plt.ylabel(
    'PC2 (18.40% variance)',
    fontsize=11
)

plt.title(
    'Customers in Principal Component Space',
    fontsize=14,
    fontweight='bold'
)

plt.legend(
    title='Customer Value Segment',
    frameon=True
)

plt.tight_layout()
plt.show()